# Light — CUDA + heavy tests on Colab

Runs Light's GPU test suites and the `LIGHT_HEAVY_TESTS=1` model tests on a Colab GPU.

**Runtime:** *Runtime → Change runtime type* → **L4** or **A100**, with **High-RAM** on.
The free **T4 will not work**: the kernels ship as `compute_80` PTX, which the driver cannot JIT for the T4's older `sm_75`.

**What it does**
1. Checks the GPU, the driver and host memory.
2. Installs Go, clones the repo, and points cgo at Colab's CUDA, cuBLAS and cuDNN.
3. Rebuilds the PTX with Colab's `nvcc` if the driver is older than the shipped PTX needs.
4. Downloads the checkpoints the tests use into `~/.cache/huggingface/hub` (the tests look there). If Drive caching is on, it keeps a copy on Google Drive so later sessions skip the download. The heavy set is about 45 GB, so you need that much free space on Drive.
5. Runs the tests in groups, one cell per group, so a session timeout only loses the group that was running. A summary table is at the end. Logs go to `/content/logs`, and to Drive when Drive caching is on.

Secrets (🔑 in the left sidebar, optional): `GITHUB_TOKEN` if the repo is private, `HF_TOKEN` for gated checkpoints.

Run the cells top to bottom. After a disconnect, run cells 1–8 again (they are quick once Go and the checkpoints are cached), then the test groups you still need.

In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/sushvibez/light"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Keep a copy of the Hugging Face cache on Google Drive so later sessions skip the downloads.
USE_DRIVE_CACHE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/light-colab"  #@param {type:"string"}
#@markdown Checkpoints to download: those used by files with `LIGHT_HEAVY_TESTS` tests, optionally also those used by the regular model tests.
MODEL_SET = "heavy"  #@param ["heavy", "heavy+standard", "none"]
#@markdown Comma-separated repos to leave out of the download, e.g. `kyutai/moshiko-candle-q8`.
SKIP_REPOS = ""  #@param {type:"string"}
#@markdown Run the CPU-only heavy tests (Whisper encoder on CPU, 2B Qwen3-ASR decoder in F32 on CPU). These are slow on Colab's few vCPUs.
RUN_CPU_HEAVY = False  #@param {type:"boolean"}
#@markdown Rebuild the PTX with Colab's nvcc even if the driver looks new enough.
FORCE_PTX_REBUILD = False  #@param {type:"boolean"}

import glob, hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request

WORK = "/content/light"
LOGS = "/content/logs"
HF_HUB = os.path.expanduser("~/.cache/huggingface/hub")  # the tests glob this path directly; $HF_HOME is ignored
CPU_HEAVY = "TestWhisperTinyCPU|TestQwen3ASRMatchesTransformersCPU"
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
os.makedirs(LOGS, exist_ok=True)


def sh(cmd, check=True, cwd=None, quiet=False):
    """Run a bash command, stream its output, and return it as a string."""
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, env=os.environ.copy(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet:
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}\n{''.join(out[-20:])}")
    return "".join(out)


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def vtuple(s):
    return tuple(int(x) for x in re.findall(r"\d+", s)[:2])


print("configured; work dir", WORK)

In [ ]:
#@title 2. Check the GPU, driver and host memory
try:
    smi = sh("nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader",
             quiet=True).strip().splitlines()[0]
except Exception:
    raise SystemExit("No GPU. Runtime → Change runtime type → L4 or A100.")

gpu_name, compute_cap, gpu_mem, driver = [x.strip() for x in smi.split(",")]
DRIVER_CUDA = vtuple(re.search(r"CUDA Version:\s*([\d.]+)", sh("nvidia-smi", quiet=True)).group(1))
print(f"GPU {gpu_name}, compute capability {compute_cap}, {gpu_mem}, driver {driver} (CUDA {DRIVER_CUDA[0]}.{DRIVER_CUDA[1]})")

if vtuple(compute_cap) < (8, 0):
    raise SystemExit(f"{gpu_name} is sm_{compute_cap.replace('.', '')}; the PTX targets compute_80, so it needs "
                     "Ampere or newer. Switch the runtime to L4 or A100.")

with open("/proc/meminfo") as f:
    mem_total_gb = int(re.search(r"MemTotal:\s+(\d+)", f.read()).group(1)) / (1 << 20)
print(f"host RAM {mem_total_gb:.0f} GB")
if mem_total_gb < 40:
    print("⚠️  Turn on High-RAM. Several heavy tests stage a whole checkpoint in host memory and "
          "skip themselves when too little is free.")

In [ ]:
#@title 3. Mount Google Drive (only if Drive caching is on)
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/hf-hub", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/logs", exist_ok=True)
    print("Drive cache at", DRIVE_DIR)
else:
    print("Drive caching off; checkpoints are downloaded again every session.")

In [ ]:
#@title 4. Install Go
releases = json.load(urllib.request.urlopen("https://go.dev/dl/?mode=json"))
rel = next(r for r in releases if r["stable"])
archive = next(f for f in rel["files"] if f["os"] == "linux" and f["arch"] == "amd64" and f["kind"] == "archive")

have = sh("/usr/local/go/bin/go env GOVERSION 2>/dev/null || true", quiet=True).strip()
if have != rel["version"]:
    url = "https://go.dev/dl/" + archive["filename"]
    print("downloading", url)
    urllib.request.urlretrieve(url, "/tmp/go.tgz")
    with open("/tmp/go.tgz", "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    if digest != archive["sha256"]:
        raise SystemExit(f"checksum mismatch for {url}")
    shutil.rmtree("/usr/local/go", ignore_errors=True)
    sh("tar -C /usr/local -xzf /tmp/go.tgz && rm /tmp/go.tgz")

os.environ["PATH"] = "/usr/local/go/bin:/root/go/bin:" + os.environ["PATH"]
os.environ["CGO_ENABLED"] = "1"
sh("go version")

In [ ]:
#@title 5. Clone the repo
url = REPO_URL
token = colab_secret("GITHUB_TOKEN")
if token and url.startswith("https://github.com/"):
    url = url.replace("https://", f"https://x-access-token:{token}@", 1)

if os.path.isdir(f"{WORK}/.git"):
    # Colab scratch checkout: discard local changes (e.g. rebuilt PTX) and move to the branch tip.
    sh(f"git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD && git clean -fdq", cwd=WORK, quiet=True)
else:
    sh(f"git clone --depth 1 --branch {BRANCH} {url} {WORK}", quiet=True)
sh("git log -1 --format='%h %s (%cd)'", cwd=WORK)

In [ ]:
#@title 6. Point cgo at CUDA, cuBLAS and cuDNN
HEADERS = ["cuda.h", "cuda_runtime.h", "cublas_v2.h", "cublasLt.h", "cudnn.h"]
LIBS = ["cuda", "cudart", "cublas", "cublasLt", "cudnn"]
LINK_DIR = "/content/cudalinks"  # unversioned lib*.so symlinks for the linker


def search_dirs():
    dirs = ["/usr/lib64-nvidia",  # Colab's real libcuda (the driver library)
            "/usr/local/cuda/include", "/usr/local/cuda/lib64",
            "/usr/local/cuda/targets/x86_64-linux/include", "/usr/local/cuda/targets/x86_64-linux/lib",
            "/usr/include", "/usr/include/x86_64-linux-gnu", "/usr/lib/x86_64-linux-gnu",
            "/usr/local/nvidia/lib64"]
    for sp in sorted({p for p in sys.path if p.endswith("-packages")}):
        dirs += sorted(glob.glob(f"{sp}/nvidia/*/include")) + sorted(glob.glob(f"{sp}/nvidia/*/lib"))
    return [d for d in dirs if os.path.isdir(d)]


def locate():
    headers, libs = {}, {}
    for d in search_dirs():
        for h in HEADERS:
            if h not in headers and os.path.isfile(f"{d}/{h}"):
                headers[h] = d
        for lib in LIBS:
            if lib not in libs:
                hits = sorted(glob.glob(f"{d}/lib{lib}.so*"))
                if hits:
                    libs[lib] = hits[0]  # the unversioned name sorts first when present
    return headers, libs


headers, libs = locate()
if "cudnn.h" not in headers or "cudnn" not in libs:
    cuda_major = vtuple(sh("/usr/local/cuda/bin/nvcc --version", quiet=True).split("release")[-1])[0]
    print(f"cuDNN not found; installing nvidia-cudnn-cu{cuda_major}")
    sh(f"pip install -q nvidia-cudnn-cu{cuda_major}")
    import site
    sys.path.extend(p for p in site.getsitepackages() if p not in sys.path)
    headers, libs = locate()

missing = [h for h in HEADERS if h not in headers] + [f"lib{l}.so" for l in LIBS if l not in libs]
if missing:
    raise SystemExit(f"still missing: {missing}")

os.makedirs(LINK_DIR, exist_ok=True)
for lib, path in libs.items():
    link = f"{LINK_DIR}/lib{lib}.so"
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(path, link)

inc_dirs = list(dict.fromkeys(headers.values()))
lib_dirs = list(dict.fromkeys(os.path.dirname(p) for p in libs.values()))
os.environ["CGO_CFLAGS"] = " ".join(f"-I{d}" for d in inc_dirs)
os.environ["CGO_LDFLAGS"] = " ".join(f"-L{d}" for d in [LINK_DIR] + lib_dirs)
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")]).rstrip(":")
for h, d in headers.items():
    print(f"{h:16} {d}")
for lib, p in libs.items():
    print(f"lib{lib + '.so':14} {p}")

In [ ]:
#@title 7. Rebuild the PTX if the driver is too old for it
# PTX ISA version → first CUDA release that can JIT it.
PTX_NEEDS_CUDA = {(8, 4): (12, 4), (8, 5): (12, 5), (8, 6): (12, 7), (8, 7): (12, 8),
                  (8, 8): (12, 9), (9, 0): (13, 0), (9, 1): (13, 1)}


def ptx_isa(path):
    with open(path) as f:
        for line in f:
            m = re.match(r"\.version\s+(\d+)\.(\d+)", line)
            if m:
                return int(m[1]), int(m[2])


ptx_files = sorted(glob.glob(f"{WORK}/kernels/cuda/ptx/*.ptx"))
shipped = max(ptx_isa(p) for p in ptx_files)
needs = PTX_NEEDS_CUDA.get(shipped)
if needs:
    print(f"shipped PTX ISA {shipped[0]}.{shipped[1]} needs CUDA {needs[0]}.{needs[1]}+; "
          f"driver supports {DRIVER_CUDA[0]}.{DRIVER_CUDA[1]}")
else:
    print(f"shipped PTX ISA {shipped[0]}.{shipped[1]} is not in the table; assuming the driver can load it")

if FORCE_PTX_REBUILD or (needs and DRIVER_CUDA < needs):
    nvcc = vtuple(sh("/usr/local/cuda/bin/nvcc --version", quiet=True).split("release")[-1])
    print(f"rebuilding PTX with nvcc {nvcc[0]}.{nvcc[1]}")
    if nvcc > DRIVER_CUDA:
        print("⚠️  nvcc is newer than the driver, so the rebuilt PTX may still be too new.")
    sh("bash scripts/build_ptx.sh", cwd=WORK, quiet=True)
    print("now PTX ISA", ".".join(map(str, max(ptx_isa(p) for p in ptx_files))))
else:
    print("no rebuild needed")

In [ ]:
#@title 8. Smoke test: cgo build, device query, one PTX kernel launch
# Compile (and link) the cgo package. Not `go vet`: its unsafeptr check flags every uintptr device pointer
# handed back to CUDA, which is a false positive for GPU addresses.
sh("go build ./internal/cudart ./device", cwd=WORK)
sh("go test -count=1 -v -run 'TestDeviceCount|TestDeviceProperties|TestLaunchAddF32' ./internal/cudart", cwd=WORK)
# If TestLaunchAddF32 fails with a PTX / JIT error, tick FORCE_PTX_REBUILD in cell 1 and re-run cells 1, 7 and 8.

In [ ]:
#@title 9. Download the checkpoints the tests use
def discover_repos():
    """Find HF repos the tests load, split by whether the file has LIGHT_HEAVY_TESTS tests."""
    consts = {}
    for path in glob.glob(f"{WORK}/**/*.go", recursive=True):
        with open(path, encoding="utf-8") as f:
            for name, repo in re.findall(r'\b(\w+Repo)\s*=\s*"([\w.-]+/[\w.-]+)"', f.read()):
                consts[name] = repo
    heavy, standard = set(), set()
    for path in glob.glob(f"{WORK}/**/*_test.go", recursive=True):
        with open(path, encoding="utf-8") as f:
            src = f.read()
        repos = {f"{org}/{name}" for org, name in re.findall(r"models--([\w.-]+?)--([\w.-]+?)/", src)}
        repos |= set(re.findall(r'FromHF\(\s*"([\w.-]+/[\w.-]+)"', src))
        repos |= {repo for name, repo in consts.items() if re.search(rf"\b{name}\b", src)}
        repos = {r for r in repos if not r.startswith("testdata/")}
        (heavy if "LIGHT_HEAVY_TESTS" in src else standard).update(repos)
    return sorted(heavy), sorted(standard - heavy)


heavy_repos, standard_repos = discover_repos()
print("heavy-test repos:   ", ", ".join(heavy_repos))
print("regular-test repos: ", ", ".join(standard_repos))

wanted = {"heavy": heavy_repos, "heavy+standard": heavy_repos + standard_repos, "none": []}[MODEL_SET]
skip = {s.strip() for s in SKIP_REPOS.split(",") if s.strip()}
wanted = [r for r in wanted if r not in skip]

sh("command -v rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync)", quiet=True)
os.makedirs(HF_HUB, exist_ok=True)
DRIVE_HUB = f"{DRIVE_DIR}/hf-hub"
if USE_DRIVE_CACHE:
    # Drive can't hold symlinks, so the Drive copy has real files under snapshots/ and no blobs/.
    print("restoring cache from Drive …")
    sh(f"rsync -rt '{DRIVE_HUB}/' '{HF_HUB}/'")

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"{free_gb:.0f} GB free on local disk; downloading {len(wanted)} repos")

from huggingface_hub import snapshot_download
IGNORE = ["*.onnx", "onnx/*", "*.msgpack", "*.h5", "flax_model*", "tf_model*", "*.ot",
          "coreml/*", "*.mlmodel", "original/*"]
failed = []
for repo in wanted:
    try:
        snapshot_download(repo, ignore_patterns=IGNORE, token=colab_secret("HF_TOKEN"))
        print("✓", repo)
    except Exception as e:
        failed.append(repo)
        print("✗", repo, "—", str(e).splitlines()[0])

if USE_DRIVE_CACHE:
    print("saving cache to Drive …")
    sh(f"rsync -rtL --exclude 'blobs/' --exclude '.locks/' '{HF_HUB}/' '{DRIVE_HUB}/'")
sh(f"du -sh '{HF_HUB}'")
if failed:
    print("⚠️  failed downloads (their tests will skip):", ", ".join(failed))

In [ ]:
#@title 10. Test runner (defines `go_test`; run this before any test group)
RESULTS = {}  # (group, package, test) -> (action, seconds, last output line mentioning a .go file)


def go_test(group, pkgs, run=None, skip=None, timeout="180m"):
    skips = [s for s in [skip] if s]
    if not RUN_CPU_HEAVY:
        skips.append(CPU_HEAVY)
    args = ["go", "test", "-json", "-count=1", f"-timeout={timeout}"]
    if run:
        args.append(f"-run={run}")
    if skips:
        args.append("-skip=" + "|".join(skips))
    args += pkgs
    env = os.environ.copy()
    env["LIGHT_HEAVY_TESTS"] = "1"
    print("$", " ".join(args), "\n")

    detail = {}
    log_path = f"{LOGS}/{group}.jsonl"
    started = time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(args, cwd=WORK, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line)
            try:
                ev = json.loads(line)
            except ValueError:
                print(line, end="")  # build errors are plain text
                continue
            act, pkg, test = ev.get("Action"), ev.get("Package", ""), ev.get("Test")
            if act == "output":
                out = ev["Output"]
                if not out.startswith(("=== RUN", "=== PAUSE", "=== CONT", "=== NAME")):
                    print(out, end="")
                if test and ".go:" in out:
                    detail[(pkg, test)] = out.strip()
            elif act in ("pass", "fail", "skip"):
                key = (group, pkg, test or "(package)")
                RESULTS[key] = (act, ev.get("Elapsed", 0.0), detail.get((pkg, test), ""))
        p.wait()

    if USE_DRIVE_CACHE:
        dest = f"{DRIVE_DIR}/logs/{RUN_STAMP}"
        os.makedirs(dest, exist_ok=True)
        shutil.copy(log_path, dest)
    mine = [v[0] for k, v in RESULTS.items() if k[0] == group and "/" not in k[2]]
    print(f"\n{group}: {mine.count('pass')} passed, {mine.count('fail')} failed, {mine.count('skip')} skipped "
          f"in {(time.time() - started) / 60:.1f} min (exit {p.returncode})")


MODEL_GROUPS = "Gemma|Phi4|Moshi|Mimi|Whisper|Qwen3ASR"
print("runner ready")

In [ ]:
#@title 11. Core packages: CUDA parity suites (no downloads needed)
core = [p for p in sh("go list ./...", cwd=WORK, quiet=True).split()
        if not re.search(r"/(models|engine|studio)$|/examples/", p)]
go_test("core", core)

In [ ]:
#@title 12. engine (Gemma 2/3/4, Phi-4-mini generation)
go_test("engine", ["./engine"])

In [ ]:
#@title 13. models: Gemma family
go_test("models-gemma", ["./models"], run="Gemma")

In [ ]:
#@title 14. models: Phi-4-mini
go_test("models-phi4", ["./models"], run="Phi4")

In [ ]:
#@title 15. models: Moshi / Mimi
go_test("models-moshi", ["./models"], run="Moshi|Mimi")

In [ ]:
#@title 16. models: speech (Whisper, Qwen3-ASR)
go_test("models-speech", ["./models"], run="Whisper|Qwen3ASR")

In [ ]:
#@title 17. models: everything else
go_test("models-rest", ["./models"], skip=MODEL_GROUPS)

In [ ]:
#@title 18. Summary
import pandas as pd

rows = [dict(group=g, package=p.removeprefix("light/"), test=t, result=a, seconds=round(s, 1), detail=d)
        for (g, p, t), (a, s, d) in RESULTS.items() if "/" not in t]
df = pd.DataFrame(rows, columns=["group", "package", "test", "result", "seconds", "detail"])
if df.empty:
    print("no results yet; run the test groups first")
else:
    display(df.pivot_table(index="group", columns="result", values="test", aggfunc="count", fill_value=0))
    fails = df[df.result == "fail"]
    print(f"\n{len(fails)} failed")
    if len(fails):
        display(fails[["group", "package", "test", "detail"]])
    heavy_skips = df[(df.result == "skip") & ~df.detail.str.contains("LIGHT_HEAVY_TESTS")]
    print(f"\n{len(heavy_skips)} skipped for reasons other than the heavy gate (missing checkpoints, memory, CUDA):")
    display(heavy_skips[["group", "package", "test", "detail"]])
    csv = f"{LOGS}/summary-{RUN_STAMP}.csv"
    df.to_csv(csv, index=False)
    if USE_DRIVE_CACHE:
        os.makedirs(f"{DRIVE_DIR}/logs/{RUN_STAMP}", exist_ok=True)
        shutil.copy(csv, f"{DRIVE_DIR}/logs/{RUN_STAMP}/")
    print("summary saved to", csv)

In [ ]:
#@title 19. Failure details (reads the saved logs; nothing re-runs)
import collections

SKIP_PREFIXES = ("=== RUN", "=== PAUSE", "=== CONT", "=== NAME")
for log_path in sorted(glob.glob(f"{LOGS}/*.jsonl")):
    group = os.path.basename(log_path).removesuffix(".jsonl")
    out, failed = collections.defaultdict(list), []
    with open(log_path) as f:
        for line in f:
            try:
                ev = json.loads(line)
            except ValueError:
                continue
            key = (ev.get("Package", "").removeprefix("light/"), ev.get("Test"))
            if not key[1]:
                continue
            if ev["Action"] == "output" and not ev["Output"].startswith(SKIP_PREFIXES):
                out[key].append(ev["Output"])
            elif ev["Action"] == "fail":
                failed.append(key)
    if not failed:
        continue
    print(f"\n######## {group}: {len(failed)} failed (top-level tests and subtests)")
    for pkg, test in failed:
        print(f"===== {pkg}  {test}")
        print("".join(out[(pkg, test)][-12:]))